```otter
# ASSIGNMENT CONFIG
name: practice09
requirements: ../p_requirements.txt
python_version: 3.12
solutions_pdf: true
export_cell: false
generate:
    pdf: true
    filtering: true
    pagebreaks: true
    zips: false
```

**Student names and e-mails:**

_YOUR NAME — your@calvin.edu_

_YOUR NAME — your@calvin.edu_

# Practice 09 — Name That Genre: Logistic Regression, Random Forests and AdaBoost

Open a playlist called "Rock Classics" and you know what to expect. But what makes a song *sound* like rock, or rap, or latin? Spotify describes every song with a handful of numbers: how danceable it is, how loud, how much of it is spoken words. Can a model hear the genre in those numbers?

This practice trains the three models from this week's classes on the same songs, and compares them.

| SLO | Description |
|-----|-------------|
| **11A** | Train and interpret logistic regression models for binary and multiclass classification |
| **11B** | Train ensemble models (random forests, gradient boosting) and explain how they improve on a single decision tree |

---
## The Dataset: 6,000 Songs, Six Genres

Kaylin Pavlik collected about 5,000 songs from each of six genres through Spotify's API, and in January 2020 the TidyTuesday project shared her data (CC0). Each song came from a playlist, and **the playlist's genre is the label**. Songs that showed up in playlists of two or more genres (like "Despacito", in pop, latin and edm playlists) were dropped. Then 1,000 songs were drawn per genre.

Spotify's software computed the audio numbers from each recording. We see the numbers, not how they were made.

| Column | What it holds |
|---|---|
| `song`, `artist` | the title and the artist |
| `year` | the album's release year (the models won't use it) |
| `danceability` | 0 to 1: how suitable for dancing (tempo, steady rhythm, strong beat) |
| `energy` | 0 to 1: intensity; energetic songs feel fast, loud and noisy |
| `loudness` | average loudness, in decibels (dB): from about −60 (quiet) to 0 (loud) |
| `speechiness` | 0 to 1: how much of the track is spoken words |
| `acousticness` | 0 to 1: how sure Spotify is that the track is acoustic |
| `instrumentalness` | 0 to 1: how sure Spotify is that the track has no vocals |
| `liveness` | 0 to 1: how sure Spotify is that an audience is present (a live recording) |
| `valence` | 0 to 1: how positive (happy, cheerful) the track sounds |
| `tempo` | beats per minute |
| `duration_min` | length, in minutes |
| `genre` | **the label**: edm, latin, pop, r&b, rap or rock |

### Setup (given — just run it)

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import confusion_matrix

songs = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/songs.csv")
print(len(songs), "songs |", songs["genre"].nunique(), "genres")
songs.head()

The features are the ten audio numbers. The split is given: 25% of the songs for testing.

In [ ]:
features = ["danceability", "energy", "loudness", "speechiness", "acousticness",
            "instrumentalness", "liveness", "valence", "tempo", "duration_min"]
X = songs[features]
y = songs["genre"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
print(len(X_train), "training songs |", len(X_test), "test songs")

`stratify=y` keeps the genres balanced: 750 training songs and 250 test songs per genre. So a lazy model that always guesses the same genre is right 1 time in 6: **16.7%**.

---
## Part 1 — Logistic Regression with Six Genres (SLO 11A)

On Monday, logistic regression had two classes: one score, squeezed into one probability. With six genres, it learns **one score per genre**, each with its own slopes. Softmax then turns the six scores into six probabilities that add up to 1.

As on Monday, the numbers are scaled first, so that every slope is "per standard deviation" of its column.

```otter
# BEGIN QUESTION
name: 11A.1
points: 2
```

### Task 11A.1 — Train and Score Logistic Regression *(2 pts)*

1. Build a pipeline that scales the columns, then runs logistic regression: `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))`. Store it in `logistic`. (`max_iter` gives the solver enough steps to settle.)
2. Score it with 5-fold cross-validation on the training songs: `cross_validate`, with `cv=5` and `return_train_score=True`. Store the result in `logistic_scores`.

The printout is given.

```otter
# BEGIN SOLUTION
```

In [ ]:
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))  # SOLUTION
logistic_scores = cross_validate(logistic, X_train, y_train, cv=5, return_train_score=True)  # SOLUTION

print(f"logistic regression | folds it learned from: {logistic_scores['train_score'].mean():.3f}"
      f" | held-out fold: {logistic_scores['test_score'].mean():.3f}")

```otter
# END SOLUTION
```

```otter
# BEGIN TESTS
```

In [ ]:
assert hasattr(logistic, "steps") and isinstance(logistic[0], StandardScaler), "make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))"
assert isinstance(logistic[-1], LogisticRegression), "the pipeline's last step should be LogisticRegression(max_iter=1000)"
assert "train_score" in logistic_scores and len(logistic_scores["test_score"]) == 5, "use cross_validate with cv=5 and return_train_score=True"
assert 0.35 < logistic_scores["test_score"].mean() < 0.6, "cross-validate on X_train and y_train"

In [ ]:
""" # BEGIN TEST CONFIG
hidden: true
""" # END TEST CONFIG

assert abs(logistic_scores["test_score"].mean() - 0.45578) < 5e-4
assert abs(logistic_scores["train_score"].mean() - 0.46389) < 5e-4

```otter
# END TESTS
```

```otter
# END QUESTION
```

About **46%** right on the held-out fold: nearly three times the lazy 16.7%, but far from perfect. The two scores are close (46.4% on the folds it learned from), so logistic regression is **not overfitting**. If anything, it **underfits**: each genre's score is a weighted sum of the columns, so a slope can only say "the faster, the more likely" (or the less likely). It can't learn "a medium tempo".

> 📌 **One split:** cross-validation averages 5 folds, but they all come from these 4,500 songs.

> 🎯 **Predict First:** Which column do you think pushes a song most **toward rap**? Which pushes a song most **away from rock**?

```otter
# BEGIN QUESTION
name: 11A.2
points: 2
```

### Task 11A.2 — Read the Slopes *(2 pts)*

1. Fit `logistic` on the training songs.
2. `logistic[-1].coef_` holds the slopes: one row per genre (in the order of `logistic.classes_`) and one column per feature. Store them as a table: `slopes = pd.DataFrame(logistic[-1].coef_, index=logistic.classes_, columns=features)`.
3. Store the name of the column with the **largest** slope for rap in `rap_top`, and the column with the **most negative** slope for rock in `rock_bottom`. (*Hint:* `slopes.loc["rap"]` is one row; `.idxmax()` and `.idxmin()` give the name of its largest and smallest entry.)

The heatmap is given.

```otter
# BEGIN SOLUTION
```

In [ ]:
logistic.fit(X_train, y_train)  # SOLUTION
slopes = pd.DataFrame(logistic[-1].coef_, index=logistic.classes_, columns=features)  # SOLUTION
rap_top = slopes.loc["rap"].idxmax()  # SOLUTION
rock_bottom = slopes.loc["rock"].idxmin()  # SOLUTION

print("pushes most toward rap:", rap_top, "| pushes most away from rock:", rock_bottom)
px.imshow(slopes.round(2), text_auto=True, aspect="auto", color_continuous_scale="RdBu", color_continuous_midpoint=0,
          labels={"color": "slope"}, title="Logistic Regression: One Row of Slopes per Genre").show()

```otter
# END SOLUTION
```

```otter
# BEGIN TESTS
```

In [ ]:
assert slopes.shape == (6, 10), "one row per genre and one column per feature: fit logistic first"
assert list(slopes.columns) == features, "use columns=features"
assert rap_top in features and rock_bottom in features, "store a column name, like 'energy'"

In [ ]:
""" # BEGIN TEST CONFIG
hidden: true
""" # END TEST CONFIG

assert rap_top == "speechiness"
assert rock_bottom == "danceability"
assert abs(slopes.loc["rock", "danceability"] - (-1.28517)) < 5e-4

```otter
# END TESTS
```

```otter
# END QUESTION
```

A positive slope (blue) raises that genre's score; a negative one (red) lowers it. Read one row at a time:

* **Rap:** speechiness, +0.83. Spoken words push hardest toward rap.
* **Rock:** danceability, −1.29, the biggest slope in the table: the more danceable the song, the less likely it is rock. Rock songs also have few spoken words (−0.85) and plenty of energy (+0.59).
* **edm:** energetic (+0.53), without vocals (+0.55) and not cheerful (valence −0.71).
* **Pop:** no slope beyond ±0.37. The model finds no sound that says "pop". Keep that in mind for Part 4.

### Six Probabilities for One Song

"Mi Gente", by J Balvin, was a worldwide latin hit in 2017. It is one of the test songs.

```otter
# BEGIN QUESTION
name: 11A.3
points: 2
```

### Task 11A.3 — What Does the Model Hear? *(2 pts)*

`mi_gente` (given) is the song's row of 10 numbers.

1. Store the model's six probabilities for this song in `probs`, as a Series indexed by genre. `predict_proba` returns one row per song and one column per genre (in the order of `logistic.classes_`), so take row `[0]`: `pd.Series(logistic.predict_proba(mi_gente)[0], index=logistic.classes_)`.
2. Store the model's guess, a single genre, in `guess`. (*Hint:* `predict` returns an array, so take element `[0]`.)

```otter
# BEGIN SOLUTION
```

In [ ]:
test_songs = songs.loc[X_test.index]                     # the test songs, with their names and genres
mi_gente = X_test[test_songs["song"] == "Mi Gente"]      # one row: the song's 10 numbers

probs = pd.Series(logistic.predict_proba(mi_gente)[0], index=logistic.classes_)  # SOLUTION
guess = logistic.predict(mi_gente)[0]  # SOLUTION

print(probs.round(3).to_string())
print("sum:", round(probs.sum(), 3), "| guess:", guess, "| playlist genre:", test_songs.loc[mi_gente.index, "genre"].iloc[0])

```otter
# END SOLUTION
```

```otter
# BEGIN TESTS
```

In [ ]:
assert len(probs) == 6 and abs(probs.sum() - 1) < 1e-6, "six probabilities, one per genre, that add up to 1"
assert guess in logistic.classes_, "guess should be one genre, like 'rock'"
assert guess == probs.idxmax(), "use logistic.predict(mi_gente)[0]"

In [ ]:
""" # BEGIN TEST CONFIG
hidden: true
""" # END TEST CONFIG

assert guess == "edm"
assert abs(probs["edm"] - 0.3131) < 5e-4
assert abs(probs["latin"] - 0.1639) < 5e-4

```otter
# END TESTS
```

```otter
# END QUESTION
```

The guess is simply the genre with the highest probability: **edm**, at 31%. Latin, the song's playlist genre, gets only 16%. All six add up to 1: softmax shares one unit of belief among the genres, and a top guess at 31% is a model that isn't sure. To the model, a loud song that is neither acoustic (0.02) nor cheerful (valence 0.31) sounds like edm.

Seven more test songs (given):

In [ ]:
famous = ["Scentless Apprentice", "Wanted Dead Or Alive", "Hypnotize", "Paparazzi",
          "Whenever, Wherever", "Aerodynamic", "In My Life - Remastered 2009"]
some = test_songs[test_songs["song"].isin(famous)]
table = pd.DataFrame(logistic.predict_proba(X_test.loc[some.index]), columns=logistic.classes_,
                     index=some["song"] + " (" + some["artist"] + ")").round(2)
table["guess"] = logistic.predict(X_test.loc[some.index])
table["playlist genre"] = some["genre"].to_numpy()
table

Nirvana and Bon Jovi are easy: 91% and 84% rock. Lady Gaga's "Paparazzi" is right, but at only 27%. The Beatles' "In My Life", quiet and acoustic, sounds like r&b to the model.

Two "mistakes" deserve a second look: Shakira's "Whenever, Wherever" and Daft Punk's "Aerodynamic" are labeled **pop** because they came from pop playlists. The model calls them latin and edm. Who is wrong here?

---
## Part 2 — From One Tree to a Forest (SLO 11B)

First, one decision tree with no depth limit (given — just run it):

In [ ]:
tree = DecisionTreeClassifier(random_state=1)
tree_scores = cross_validate(tree, X_train, y_train, cv=5, return_train_score=True)
print(f"one tree, no depth limit | folds it learned from: {tree_scores['train_score'].mean():.3f}"
      f" | held-out fold: {tree_scores['test_score'].mean():.3f}")

A clear case of **overfitting**: the tree memorizes the songs it learned from (100%) but gets only 38% of the held-out songs, worse than logistic regression.

```otter
# BEGIN QUESTION
name: 11B.1
points: 2
```

### Task 11B.1 — A Random Forest *(2 pts)*

1. Build a `RandomForestClassifier` with `n_estimators=100` and `random_state=1`. Store it in `forest`. (No scaling needed: a tree only compares one column with a cutoff.)
2. Score it as in Task 11A.1, and store the result in `forest_scores`. (This takes a few seconds.)

```otter
# BEGIN SOLUTION
```

In [ ]:
forest = RandomForestClassifier(n_estimators=100, random_state=1)  # SOLUTION
forest_scores = cross_validate(forest, X_train, y_train, cv=5, return_train_score=True)  # SOLUTION

print(f"random forest, 100 trees | folds it learned from: {forest_scores['train_score'].mean():.3f}"
      f" | held-out fold: {forest_scores['test_score'].mean():.3f}")

```otter
# END SOLUTION
```

```otter
# BEGIN TESTS
```

In [ ]:
assert isinstance(forest, RandomForestClassifier), "use RandomForestClassifier"
assert forest.n_estimators == 100 and forest.random_state == 1, "use n_estimators=100 and random_state=1"
assert "train_score" in forest_scores and len(forest_scores["test_score"]) == 5, "use cross_validate with cv=5 and return_train_score=True"

In [ ]:
""" # BEGIN TEST CONFIG
hidden: true
""" # END TEST CONFIG

assert abs(forest_scores["test_score"].mean() - 0.51733) < 5e-4

```otter
# END TESTS
```

```otter
# END QUESTION
```

Each of the 100 trees still memorizes its songs, so the forest also scores 100% on the folds it learned from. But on the held-out fold it gets **52%**: 14 points more than one tree, and 6 more than logistic regression.

Why? Each tree learns from its own bootstrap sample of songs, and looks at a few random columns at each split. So the trees make **different** mistakes, and the vote cancels many of them. The forest still overfits, but much less than one tree.

---
## Part 3 — AdaBoost (SLO 11B)

AdaBoost builds its trees **in a row**: each new tree gives more weight to the songs that the earlier trees got wrong, and the better trees get a bigger say in the final vote. Its default tree is a **stump**: a tree with a single split.

```otter
# BEGIN QUESTION
name: 11B.2
points: 2
```

### Task 11B.2 — Boosting Stumps *(2 pts)*

1. Build an `AdaBoostClassifier` with `n_estimators=100` and `random_state=1`. Store it in `ada`.
2. Score it as before, and store the result in `ada_scores`.

```otter
# BEGIN SOLUTION
```

In [ ]:
ada = AdaBoostClassifier(n_estimators=100, random_state=1)  # SOLUTION
ada_scores = cross_validate(ada, X_train, y_train, cv=5, return_train_score=True)  # SOLUTION

print(f"AdaBoost, 100 stumps | folds it learned from: {ada_scores['train_score'].mean():.3f}"
      f" | held-out fold: {ada_scores['test_score'].mean():.3f}")

```otter
# END SOLUTION
```

```otter
# BEGIN TESTS
```

In [ ]:
assert isinstance(ada, AdaBoostClassifier), "use AdaBoostClassifier"
assert ada.n_estimators == 100 and ada.random_state == 1, "use n_estimators=100 and random_state=1"
assert "train_score" in ada_scores and len(ada_scores["test_score"]) == 5, "use cross_validate with cv=5 and return_train_score=True"

In [ ]:
""" # BEGIN TEST CONFIG
hidden: true
""" # END TEST CONFIG

assert abs(ada_scores["test_score"].mean() - 0.455) < 0.01

```otter
# END TESTS
```

```otter
# END QUESTION
```

About **46%**, no better than logistic regression, and close to its score on the folds it learned from (48%): **underfitting**. A stump splits the songs into just two groups, so each stump names at most two of the six genres. A hundred of them, each that simple, can't sort six overlapping genres. (Reading 9 found the same with seven tree species.)

AdaBoost can also boost bigger trees, for example `AdaBoostClassifier(DecisionTreeClassifier(max_depth=4), ...)`, and then the depth of each tree becomes one more knob to tune.

---
## Part 4 — The Final Comparison (SLO 11B)

Cross-validation ranked the forest first. Now, **once**, open the test set.

```otter
# BEGIN QUESTION
name: 11B.3
points: 2
```

### Task 11B.3 — Test Accuracy *(2 pts)*

Fit each model, `logistic`, `forest` and `ada`, on all the training songs, and store its accuracy on the test songs in `logistic_test`, `forest_test` and `ada_test`. (*Hint:* `model.fit(X_train, y_train).score(X_test, y_test)` does both in one line.)

The summary table is given.

```otter
# BEGIN SOLUTION
```

In [ ]:
logistic_test = logistic.fit(X_train, y_train).score(X_test, y_test)  # SOLUTION
forest_test = forest.fit(X_train, y_train).score(X_test, y_test)  # SOLUTION
ada_test = ada.fit(X_train, y_train).score(X_test, y_test)  # SOLUTION

pd.DataFrame({"CV: folds it learned from": [logistic_scores["train_score"].mean(), forest_scores["train_score"].mean(), ada_scores["train_score"].mean()],
              "CV: held-out fold": [logistic_scores["test_score"].mean(), forest_scores["test_score"].mean(), ada_scores["test_score"].mean()],
              "test songs": [logistic_test, forest_test, ada_test]},
             index=["logistic regression", "random forest", "AdaBoost, 100 stumps"]).round(3)

```otter
# END SOLUTION
```

```otter
# BEGIN TESTS
```

In [ ]:
assert 0.3 < logistic_test < 0.7 and 0.3 < forest_test < 0.7 and 0.3 < ada_test < 0.7, "use .score(X_test, y_test) after fitting on X_train, y_train"
assert forest_test > logistic_test, "fit each model on the training songs, then score it on the test songs"

In [ ]:
""" # BEGIN TEST CONFIG
hidden: true
""" # END TEST CONFIG

assert abs(logistic_test - 0.468) < 5e-4
assert abs(forest_test - 0.52467) < 5e-4
assert 0.43 < ada_test < 0.50

```otter
# END TESTS
```

```otter
# END QUESTION
```

The test set agrees with cross-validation: the forest first (52.5%), then logistic regression (46.8%) and AdaBoost (about 45%).

> 📌 **One split:** these are 1,500 test songs. Another split would move each score by a point or two.

Which genres does the forest mix up? (given)

In [ ]:
genres = forest.classes_
counts = confusion_matrix(y_test, forest.predict(X_test), labels=genres)
px.imshow(counts, x=genres, y=genres, text_auto=True, color_continuous_scale="Blues",
          labels={"x": "the forest's guess", "y": "playlist genre", "color": "songs"},
          title="The Forest on 1,500 Test Songs (250 per Genre)").show()

The diagonal holds the right guesses. Rock (182 of 250), edm (166) and rap (151) are the easiest. **Pop** is the hardest: the forest calls only 76 of its 250 songs pop, and spreads the rest over rock (49), r&b (46), latin (33) and edm (32). Logistic regression found no slope for pop, either.

Part of the problem is the label. A song's genre here is the genre of the **playlist** it came from, and a person chose it. Daft Punk sits in a pop playlist; so does Shakira. And 7% of the songs Pavlik collected sat in playlists of two genres or more. As Deborah Stone argues in the first chapter of *Counting*, putting things in a category means deciding which differences to ignore. No model can be more consistent than the labels it learns from.

---
## Submission

Save your notebook, then upload the **`.ipynb` file** directly to our Moodle assignment page. Don't export, zip, or submit anything else — just the notebook.